# Lab 8 · LLMs para analizar reclamaciones y tickets
### Tema 8 · MU Gestión de Sistemas Complejos · Caso transversal: operador de telecomunicaciones

**Entorno:** SageMaker Notebook (kernel Python 3) · **Librerías:** pandas (+ boto3 si usas Bedrock)
**Duración estimada:** 150–180 min · **Nivel:** intermedio

---

## 1. Idea general del laboratorio

Volvemos al tipo de traza más rico y difícil: el **texto libre** (tickets, reclamaciones). Extraeremos
información **estructurada** —tema, servicio, urgencia, sentimiento, causa— convirtiendo una columna de
texto en columnas analizables. Consigna: **un LLM es potente pero falible** (alucina, no capta ironía,
sesga). Por eso diseñaremos prompts que limiten la invención, exigiremos JSON y **validaremos contra una
muestra revisada a mano**.

### Dos caminos en este notebook
- **Bedrock (opcional):** si tienes un `modelId` habilitado, pon `USAR_BEDROCK = True`.
- **PLN clásico (por defecto, siempre ejecutable):** un extractor por **reglas y palabras clave**. Es peor
  que un LLM —y esa diferencia es material de reflexión—, pero hace que el notebook corra **de principio a
  fin** en un SageMaker Notebook estándar, sin coste de tokens. La función `extraer()` elige el camino
  disponible automáticamente.

> **Nota para el vídeo.** Cada celda está comentada paso a paso. Datos: `telco_tickets_text_lab8.csv` (120
> tickets) y `mi_anotacion.csv` (10 anotados a mano), incluidos en el paquete.

In [ ]:
# ===========================================================================
# CELDA 1 · Configuracion y carga de datos
# ===========================================================================
import pandas as pd, json, time, re, unicodedata

# Pon True solo si tienes un modelId de Bedrock habilitado en tu entorno AWS.
USAR_BEDROCK = False
MODEL_ID = 'amazon.titan-text-express-v1'   # sustituye por el de tu cuenta

# RUTA local por defecto; para S3 usa 's3://<tu-bucket>/text'.
RUTA = '.'
df = pd.read_csv(f'{RUTA}/telco_tickets_text_lab8.csv')
print("Tickets:", df.shape)
df.head(3)

## Parte A · Primera llamada a Bedrock (opcional)

Patrón general con `bedrock-runtime`. Si `USAR_BEDROCK = False`, esta celda no invoca nada.

> **Buena práctica.** `temperature = 0` => respuestas deterministas. En extracción queremos consistencia,
> no creatividad.

In [ ]:
# ===========================================================================
# CELDA 2 · Cliente Bedrock y funcion preguntar() (solo si USAR_BEDROCK)
# ===========================================================================
bedrock = None
if USAR_BEDROCK:
    import boto3
    # Cliente del runtime de inferencia de Bedrock en la region us-east-1.
    bedrock = boto3.client('bedrock-runtime', region_name='us-east-1')

    def preguntar(prompt, max_tokens=512):
        # 'cuerpo' es el payload que espera la familia Titan: texto + config.
        cuerpo = {
            "inputText": prompt,
            "textGenerationConfig": {"maxTokenCount": max_tokens, "temperature": 0.0}
        }
        # invoke_model envia el prompt; body debe ir serializado a JSON.
        resp = bedrock.invoke_model(modelId=MODEL_ID, body=json.dumps(cuerpo))
        # La respuesta llega como bytes JSON: la parseamos y extraemos el texto.
        out = json.loads(resp['body'].read())
        return out['results'][0]['outputText']

    # Prueba de humo: comprobamos que el modelo responde.
    try:
        print(preguntar("Responde solo OK si me lees."))
    except Exception as e:
        print("Bedrock no disponible, se usara el extractor clasico. Detalle:", e)
        USAR_BEDROCK = False
else:
    print("Bedrock desactivado (USAR_BEDROCK=False). Se usara el extractor clasico de PLN.")

## Parte B · Diseñar el prompt de extracción

El núcleo del lab. Un buen prompt (1) define rol y tarea, (2) fija los campos y sus **valores permitidos**,
(3) exige **JSON**, y (4) **prohíbe inventar**.

In [ ]:
# ===========================================================================
# CELDA 3 · Plantilla de prompt para extraccion estructurada
# ===========================================================================
# {texto} es un hueco que rellenaremos con cada ticket (con .format()).
# Las claves y sus valores permitidos van EXPLICITOS para forzar salida
# verificable; la ultima linea prohibe alucinar causas que el texto no dice.
PLANTILLA = """Eres un analista de atencion al cliente de una operadora.
Extrae del siguiente ticket la informacion pedida y responde SOLO con un
objeto JSON, sin texto adicional, con estas claves exactas:

- tema: uno de [averia, velocidad, facturacion, portabilidad, instalacion,
  consulta, baja, felicitacion, otro]
- servicio_afectado: uno de [internet, movil, fijo, television, varios, ninguno]
- urgencia: uno de [baja, media, alta, critica]
- sentimiento: uno de [positivo, neutral, negativo]
- posible_causa: texto breve SOLO si aparece explicitamente en el ticket;
  si no aparece, escribe "no especificada".

No inventes informacion que no este en el texto.

TICKET:
\"\"\"{texto}\"\"\"
"""

# Vemos como queda el prompt con el primer ticket insertado.
print(PLANTILLA.format(texto=df.iloc[0]['text']))

## Parte B-bis · Extractor clásico de PLN (camino sin Bedrock)

Replicamos la tarea con **reglas y diccionarios**. Es más simple y **fallará con la ironía** —y ese fallo
es justo lo que el entregable debe comentar como límite frente a un LLM.

In [ ]:
# ===========================================================================
# CELDA 4 · Diccionarios de palabras clave y normalizacion de texto
# ===========================================================================
def _norm(t):
    # Pasa a minusculas y QUITA acentos (NFKD + encode ascii) para que las
    # comparaciones no fallen por tildes ("averia" == "averia").
    t = t.lower()
    t = unicodedata.normalize('NFKD', t).encode('ascii', 'ignore').decode('ascii')
    return t

# Para cada TEMA, lista de expresiones que lo delatan.
KW_TEMA = {
    'averia':       ['sin internet', 'sin linea', 'no funciona', 'averia', 'corta',
                     'parpadea', 'sin cobertura', 'congelada', 'cuadraditos', 'no da tono'],
    'velocidad':    ['lenta', 'lentisima', 'megas', 'velocidad', 'va lento', 'penosa'],
    'facturacion':  ['factura', 'cobrado', 'cobro', 'recibo', 'cargo', 'cuota', 'euros', 'premium'],
    'portabilidad': ['portabilidad', 'portar', 'traer mi numero', 'numero de movil desde'],
    'instalacion':  ['instalacion', 'tecnico', 'instalar', 'cita', 'router esta puesto'],
    'baja':         ['baja', 'cancelacion', 'cancelad', 'darme de baja', 'me cambio a otra'],
    'felicitacion': ['agradecer', 'gracias', 'contento', 'excelente', 'un diez', 'perfecto el tecnico'],
    'consulta':     ['tarifas', 'consultar', 'duda', 'cobertura de fibra en mi', 'horario',
                     'permanencia', 'puedo anadir', 'me gustaria conocer', 'querria saber'],
    'otro':         ['direccion de correo', 'titular del contrato', 'contrasena', 'area de cliente'],
}
# Para cada SERVICIO afectado, sus palabras tipicas.
KW_SERVICIO = {
    'internet':   ['internet', 'fibra', 'router', 'megas', 'conexion'],
    'movil':      ['movil', 'cobertura', 'datos del movil', 'linea de movil', '5g'],
    'television': ['television', 'tele', 'canales', 'futbol'],
    'fijo':       ['telefono fijo', 'fijo', 'tono', 'llamadas premium'],
}
# Lexicos de sentimiento (la ironia NO se detecta -> limite conocido).
NEG = ['sin internet', 'no funciona', 'harto', 'inadmisible', 'penosa', 'lentisima',
       'no entiendo', 'reclamo', 'nadie me coge', 'retraso', 'queja', 'mal', 'sin servicio',
       'no puedo', 'congelada', 'sin linea', 'sin cobertura', 'culpa']
POS = ['gracias', 'agradecer', 'contento', 'excelente', 'perfecto', 'un diez', 'amable', 'rapidisimo']
# Senales de urgencia alta / critica.
URG_ALTA = ['urgente', 'inmediatamente', 'cuanto antes', 'sin servicio', 'sin internet',
            'sin linea', 'sin cobertura', 'no puedo ni llamar', 'teletrabajo', 'dia libre para nada']
URG_CRIT = ['averia general', 'todo', 'tres dias sin', 'inadmisible']
print("Diccionarios cargados")

In [ ]:
# ===========================================================================
# CELDA 5 · Funcion extractora basada en reglas
# ===========================================================================
def extraer_clasico(texto):
    t = _norm(texto)

    # --- TEMA: el grupo de palabras clave con mas coincidencias gana ---
    mejor, score = 'otro', 0
    for tema, kws in KW_TEMA.items():
        s = sum(1 for k in kws if _norm(k) in t)   # nº de aciertos del tema
        if s > score:
            mejor, score = tema, s
    tema = mejor

    # --- SERVICIO: idem; si aparecen >=3 servicios distintos -> 'varios' ---
    serv, ssc = 'ninguno', 0
    for sv, kws in KW_SERVICIO.items():
        s = sum(1 for k in kws if _norm(k) in t)
        if s > ssc:
            serv, ssc = sv, s
    if sum(1 for sv, kws in KW_SERVICIO.items()
           if any(_norm(k) in t for k in kws)) >= 3:
        serv = 'varios'

    # --- SENTIMIENTO: cuenta de palabras positivas vs negativas ---
    # (la ironia engana a este metodo: "perfecto, llevo una semana sin internet")
    npos = sum(1 for w in POS if _norm(w) in t)
    nneg = sum(1 for w in NEG if _norm(w) in t)
    if npos > nneg:
        sent = 'positivo'
    elif nneg > npos:
        sent = 'negativo'
    else:
        sent = 'neutral'

    # --- URGENCIA: por senales lexicas; consultas/felicitaciones -> baja ---
    if any(_norm(w) in t for w in URG_CRIT):
        urg = 'critica'
    elif any(_norm(w) in t for w in URG_ALTA):
        urg = 'alta'
    elif tema in ('consulta', 'felicitacion', 'otro'):
        urg = 'baja'
    else:
        urg = 'media'

    # El extractor por reglas no infiere causas -> "no especificada".
    return {'tema': tema, 'servicio_afectado': serv, 'urgencia': urg,
            'sentimiento': sent, 'posible_causa': 'no especificada'}

# Prueba con el primer ticket.
print(extraer_clasico(df.iloc[0]['text']))

## Parte C · Procesar un lote de tickets

`extraer()` usa Bedrock si está activo y, si no, el extractor clásico. Procesamos los primeros 30 tickets.

In [ ]:
# ===========================================================================
# CELDA 6 · Funcion despachadora extraer() y procesado del lote
# ===========================================================================
def extraer(texto):
    if USAR_BEDROCK:
        # Camino LLM: pedimos al modelo y localizamos el JSON en su respuesta.
        salida = preguntar(PLANTILLA.format(texto=texto))
        ini, fin = salida.find("{"), salida.rfind("}")   # primer { y ultimo }
        try:
            return json.loads(salida[ini:fin + 1])        # parseo del JSON
        except Exception:
            # Si el modelo devolvio algo no parseable, marcamos error de parseo.
            return {"tema": "error", "servicio_afectado": "error", "urgencia": "error",
                    "sentimiento": "error", "posible_causa": "error_parseo"}
    else:
        # Camino clasico (siempre disponible).
        return extraer_clasico(texto)

# Procesamos los primeros 30 tickets (como en el lab original).
muestra = df.head(30).copy()
resultados = []
for _, fila in muestra.iterrows():
    resultados.append(extraer(fila['text']))
    if USAR_BEDROCK:
        time.sleep(0.3)   # pausa para evitar throttling (solo con Bedrock)

# Convertimos la lista de dicts en DataFrame y lo pegamos a los datos del ticket.
ext = pd.DataFrame(resultados)
salida = pd.concat([muestra[['ticket_id', 'region', 'text']].reset_index(drop=True),
                    ext], axis=1)
salida.head(10)

In [ ]:
# ===========================================================================
# CELDA 7 · Consolidar: convertir el texto en analitica
# ===========================================================================
# Acabamos de transformar texto libre en columnas. Ahora agregamos:
print("== Temas ==")
print(salida['tema'].value_counts())          # tema mas frecuente
print("\n== Urgencia ==")
print(salida['urgencia'].value_counts())       # distribucion de urgencia
print("\n== Sentimiento por region ==")
# unstack convierte el conteo en una tabla region x sentimiento (legible).
print(salida.groupby('region')['sentimiento'].value_counts().unstack(fill_value=0))

## Parte D · Validar contra revisión humana

**Ningún resultado de un LLM se usa sin validar.** Comparamos con `mi_anotacion.csv` (10 tickets anotados).

In [ ]:
# ===========================================================================
# CELDA 8 · % de acuerdo modelo vs humano
# ===========================================================================
humano = pd.read_csv(f'{RUTA}/mi_anotacion.csv')
# merge une la extraccion con la anotacion por 'ticket_id' (quedan los 10 comunes).
comparado = salida.merge(humano, on='ticket_id')

# Para cada campo, proporcion de coincidencias entre modelo y humano.
acierto_tema = (comparado['tema'] == comparado['tema_humano']).mean()
acierto_sent = (comparado['sentimiento'] == comparado['sentimiento_humano']).mean()
acierto_urg  = (comparado['urgencia'] == comparado['urgencia_humana']).mean()
print(f"Acuerdo en tema:        {acierto_tema:.0%}")
print(f"Acuerdo en sentimiento: {acierto_sent:.0%}")
print(f"Acuerdo en urgencia:    {acierto_urg:.0%}")

In [ ]:
# ===========================================================================
# CELDA 9 · Ver DONDE discrepan (aqui afloran las ironias mal leidas)
# ===========================================================================
# Filtramos los tickets donde el modelo y el humano NO coinciden en tema o
# sentimiento, y mostramos ambos para discutirlos en el video.
discrepa = comparado[(comparado['tema'] != comparado['tema_humano']) |
                     (comparado['sentimiento'] != comparado['sentimiento_humano'])]
for _, r in discrepa.iterrows():
    print("·", r['ticket_id'], "|", r['text'][:80])
    print("   modelo:", r['tema'], "/", r['sentimiento'],
          "  humano:", r['tema_humano'], "/", r['sentimiento_humano'])

> **Atención.** Errores típicos a buscar: **alucinación** (inventar una causa que el texto no dice),
> **ironía mal leída** ("qué servicio tan maravilloso, llevo una semana sin internet" → no es positivo),
> **sobre-urgencia** (marcar crítica una consulta), **categoría forzada**. El extractor clásico falla con
> la ironía porque ve "gracias"/"perfecto"; cuantificar esa diferencia es parte del entregable.

## Parte E · (Opcional) Knowledge Bases y RAG

Bedrock **Knowledge Bases** permite preguntas/respuestas sobre tus documentos con **RAG**: el modelo busca
primero los fragmentos relevantes y responde sobre ellos, reduciendo alucinaciones. Conceptual y opcional.

## Parte F · Entregable · Parte G · Cierre

Entregable (5–7 págs.): prompt final y decisiones de diseño; tabla de ≥30 tickets; análisis consolidado;
validación de 10 tickets (% acuerdo); **tres errores** explicados; reflexión sobre riesgos (privacidad,
sesgo, alucinación, dependencia); integración con el sistema de priorización del Lab 3. Si usaste el
extractor clásico, **compara** con lo que esperarías de un LLM. Al terminar, **Stop**; con Bedrock recuerda
que **cobra por tokens**.

---

### Cierre didáctico
Has convertido texto libre en información analizable y, sobre todo, has aprendido a **no fiarte**: exigir
formatos verificables y validar contra el juicio humano. El Lab 9 da el paso a la **causalidad**.